# Combined Deity Analysis: Truth Comparison and Scoring

This notebook contains two main analyses:
1. **Truth Comparison**: Compare predicted deities against ground truth, calculate matching metrics
2. **Scoring Analysis**: Detailed statistical analysis of deity predictions including category distributions, regional metrics, and correlations

All utility functions are imported from `evaluation_utils.py`.

## Setup and Imports

In [ ]:
# Import required libraries
import ast
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Import utility functions
from evaluation_utils import (
    calculate_fp_fn_tp,
    count_cat_types,
    evaluate_gender_accuracy_wide,
    evaluate_gender_matches_wide,
    evaluate_row,
    evaluate_subset,
    extract_matched,
    parse_comma_column,
    plot_category_comparison,
    plot_regional_metrics,
    save_df_as_png,
)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
%matplotlib inline

## Configuration

In [ ]:
# File paths - UPDATE THESE WITH YOUR FILE PATHS
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from config import DATA_INTERMEDIATE

# Hand-labeled ground truth incl. eHRAF text (not distributable; see README)
TRUTH_PATH = f"{DATA_INTERMEDIATE}/250_sampled_rows_labeled.csv"
# Merged output of 01_run_pipeline.ipynb
PREDICTION_PATH = f"{DATA_INTERMEDIATE}/pipeline_results/deity_analysis_complete.csv"
FULL_DF_PATH = f"{DATA_INTERMEDIATE}/pipeline_results/deity_analysis_complete.csv"
OUTPUT_DIR = f"{DATA_INTERMEDIATE}/evaluation/"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

---
# Part 1: Truth Comparison Analysis
---

## Load Data for Truth Comparison

In [ ]:
# Load ground truth and predictions
truth_df = pd.read_csv(TRUTH_PATH, encoding="utf-8-sig")
final_df = pd.read_csv(PREDICTION_PATH, encoding="utf-8-sig")

print(f"Ground truth shape: {truth_df.shape}")
print(f"Predictions shape: {final_df.shape}")

# Normalize gender column
final_df["gender"] = (
    final_df["gender"].str.strip().str.lower().replace({"male & female": "general"})
)

In [ ]:
# Preview data
print("Ground Truth Sample:")
display(truth_df.head())

print("\nPredictions Sample:")
display(final_df.head())

## Evaluate Deity Matching

In [ ]:
# Evaluate each row
results = []
for _, truth_row in truth_df.iterrows():
    uuid = truth_row["uuid"]
    final_row = final_df[final_df["uuid"] == uuid]
    if final_row.empty:
        continue
    result = evaluate_row(uuid, truth_row, final_row.iloc[0])
    results.append(result)

# Create results dataframe
results_df = pd.DataFrame(results)
print(f"Evaluated {len(results_df)} rows")

# Save results
matching_results_path = os.path.join(OUTPUT_DIR, "deity_matching_results.csv")
results_df.to_csv(matching_results_path, index=False, encoding="utf-8-sig")
print(f"Results saved to {matching_results_path}")

In [ ]:
# Preview matching results
results_df.head()

## Calculate Overall Metrics

In [ ]:
# Calculate overall metrics
overall_metrics = evaluate_subset(results_df)
overall_metrics["Comparison"] = "Overall"

metrics_df = pd.DataFrame([overall_metrics])
metrics_df = metrics_df[
    [
        "Comparison",
        "TP",
        "FP",
        "FN",
        "Matched Missing",
        "Precision",
        "Recall",
        "F1",
        "Accuracy",
        "Truth_Deities",
        "GPT_Deities",
    ]
]

print("\nOverall Performance Metrics:")
print("=" * 80)
display(metrics_df)

# Save metrics
save_df_as_png(metrics_df, os.path.join(OUTPUT_DIR, "overall_metrics.png"))

## Analyze by Ambiguity (if available)

In [ ]:
# Check if ambiguous column exists
if "Ambiguous" in truth_df.columns:
    # Split by ambiguity
    ambiguous_1_df = results_df[
        results_df["uuid"].isin(truth_df[truth_df["Ambiguous"] == 1]["uuid"])
    ]
    ambiguous_0_df = results_df[
        results_df["uuid"].isin(truth_df[truth_df["Ambiguous"] == 0]["uuid"])
    ]

    # Evaluate both groups
    amb_1_metrics = evaluate_subset(ambiguous_1_df)
    amb_1_metrics["Comparison"] = "Ambiguous = 1"

    amb_0_metrics = evaluate_subset(ambiguous_0_df)
    amb_0_metrics["Comparison"] = "Ambiguous = 0"

    # Combine and display
    amb_summary_df = pd.DataFrame([amb_0_metrics, amb_1_metrics])
    amb_summary_df = amb_summary_df[
        [
            "Comparison",
            "TP",
            "FP",
            "FN",
            "Matched Missing",
            "Precision",
            "Recall",
            "F1",
            "Accuracy",
            "Truth_Deities",
            "GPT_Deities",
        ]
    ]

    print("\nMetrics by Ambiguity:")
    print("=" * 80)
    display(amb_summary_df)

    # Save
    save_df_as_png(amb_summary_df, os.path.join(OUTPUT_DIR, "ambiguous_metrics.png"))
else:
    print("No 'Ambiguous' column found in ground truth data")

## Gender Matching Analysis

In [ ]:
# Prepare comparison combinations
combinations = [("Overall", truth_df, final_df)]

if "Ambiguous" in truth_df.columns:
    truth_amb_0 = truth_df[truth_df["Ambiguous"] == 0]
    truth_amb_1 = truth_df[truth_df["Ambiguous"] == 1]
    final_amb_0 = final_df[final_df["uuid"].isin(truth_amb_0["uuid"])]
    final_amb_1 = final_df[final_df["uuid"].isin(truth_amb_1["uuid"])]

    combinations.extend(
        [("Ambiguous = 0", truth_amb_0, final_amb_0), ("Ambiguous = 1", truth_amb_1, final_amb_1)]
    )

In [ ]:
# Evaluate gender matches (deity identification)
gender_wide_summary = []

for name, t_df, f_df in combinations:
    row = evaluate_gender_matches_wide(t_df, f_df)
    row["Comparison"] = name
    gender_wide_summary.append(row)

gender_wide_df = pd.DataFrame(gender_wide_summary)
cols = ["Comparison"] + [col for col in gender_wide_df.columns if col != "Comparison"]
gender_wide_df = gender_wide_df[cols]

print("\nGender Matching Performance (Deity Identification):")
print("=" * 80)
display(gender_wide_df)

save_df_as_png(gender_wide_df, os.path.join(OUTPUT_DIR, "gender_matching.png"))

In [ ]:
# Evaluate gender accuracy (gender classification for matched deities)
gender_accuracy_summary = []

for name, t_df, f_df in combinations:
    row = evaluate_gender_accuracy_wide(t_df, f_df)
    row["Comparison"] = name
    gender_accuracy_summary.append(row)

gender_accuracy_df = pd.DataFrame(gender_accuracy_summary)
cols = ["Comparison"] + [col for col in gender_accuracy_df.columns if col != "Comparison"]
gender_accuracy_df = gender_accuracy_df[cols]

print("\nGender Classification Accuracy (For Matched Deities):")
print("=" * 80)
display(gender_accuracy_df)

save_df_as_png(gender_accuracy_df, os.path.join(OUTPUT_DIR, "gender_accuracy.png"))

---
# Part 2: Scoring and Statistical Analysis
---

## Load Full Dataset for Scoring

In [ ]:
# Load full prediction dataset
full_df = pd.read_csv(FULL_DF_PATH, encoding="utf-8-sig")

print(f"Full dataset shape: {full_df.shape}")
print(f"\nColumns: {full_df.columns.tolist()}")

# Parse deities in full_df
full_df["parsed_deities"] = full_df["deities"].dropna().apply(parse_comma_column)

## Total Deity Counts

In [ ]:
# Filter valid rows (excluding "missing" deities from truth)
valid_filtered_df = truth_df[
    ~truth_df["Deities"].fillna("").str.strip().str.lower().eq("missing")
].copy()

# Extract matched deities
valid_filtered_df["matched_list"] = valid_filtered_df["Deities"].apply(extract_matched)
valid_filtered_df["truth_list"] = (
    valid_filtered_df["Deities"]
    .fillna("")
    .apply(lambda x: [i.strip().strip('"') for i in x.split(",") if i.strip()])
)
valid_filtered_df["num_truth"] = valid_filtered_df["truth_list"].apply(len)
valid_filtered_df["num_matched"] = valid_filtered_df["matched_list"].apply(len)

# Calculate totals
num_deities_total = valid_filtered_df["num_truth"].sum()
num_deities_matched = valid_filtered_df["num_matched"].sum()
total_deities_in_full_df = (
    full_df["parsed_deities"].apply(lambda x: len(x) if isinstance(x, list) else 0).sum()
)

print("\nDeity Count Summary:")
print("=" * 80)
print(f"Total deities in ground truth: {num_deities_total}")
print(f"Total deities in predictions: {total_deities_in_full_df}")
print(f"Total matched deities: {num_deities_matched}")
print(f"Match rate: {num_deities_matched / num_deities_total * 100:.2f}%")

## Gender Distribution

In [ ]:
# Gender distribution in ground truth
if "truth_genders" in results_df.columns:
    gender_expanded = results_df["truth_genders"].dropna().apply(lambda x: str(x).split(","))
    gender_list = [g.strip().lower() for sub in gender_expanded for g in sub]
    gender_counts = pd.Series(gender_list).value_counts()

    print("\nGender Distribution (Ground Truth):")
    print("=" * 80)
    print(gender_counts)

    # Visualize
    plt.figure(figsize=(10, 6))
    gender_counts.plot(kind="bar")
    plt.title("Gender Distribution in Ground Truth")
    plt.xlabel("Gender")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "gender_distribution.png"))
    plt.show()

## False Positives/Negatives by Gender

In [ ]:
# Calculate FP and FN flags
results_df["FN"] = results_df["score"].apply(lambda x: 1 if 0 in ast.literal_eval(str(x)) else 0)
results_df["FP"] = results_df["score"].apply(
    lambda x: 1 if len(ast.literal_eval(str(x))) > sum(ast.literal_eval(str(x))) else 0
)

# FN by gender
fn_gender = (
    results_df[results_df["FN"] == 1]["truth_genders"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)
print("\nFalse Negatives by Gender:")
print(fn_gender)

# FP by gender
fp_gender = (
    results_df[results_df["FP"] == 1]["final_genders"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)
print("\nFalse Positives by Gender:")
print(fp_gender)

## Category Distribution Comparison

In [ ]:
# Compare category distributions with statistical tests
cat_cols = [col for col in full_df.columns if col.startswith("cat_") and col != "cat_type"]

if cat_cols and all(col in truth_df.columns for col in cat_cols):
    results_cat_df = plot_category_comparison(
        truth_df=truth_df,
        pred_df=full_df,
        output_path=os.path.join(OUTPUT_DIR, "category_comparison.png"),
        cat_cols=cat_cols,
    )

    print("\nCategory Comparison Results:")
    print("=" * 80)
    display(results_cat_df)

    # Save results
    results_cat_df.to_csv(os.path.join(OUTPUT_DIR, "category_stats.csv"), index=False)
else:
    print("Category columns not found or not matching between datasets")

## Deity Type Classification (Individual vs Multiple)

In [ ]:
# Count cat_type distributions
if "cat_type" in full_df.columns and "cat_type" in truth_df.columns:
    cat_type_individual_full, cat_type_multiple_full = count_cat_types(full_df)
    cat_type_individual_truth, cat_type_multiple_truth = count_cat_types(truth_df)

    print("\nDeity Type Classification:")
    print("=" * 80)
    print("\nGround Truth:")
    print(f"  Individual: {cat_type_individual_truth}")
    print(f"  Multiple: {cat_type_multiple_truth}")
    print("\nPredictions:")
    print(f"  Individual: {cat_type_individual_full}")
    print(f"  Multiple: {cat_type_multiple_full}")

    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Ground truth
    ax1.bar(["Individual", "Multiple"], [cat_type_individual_truth, cat_type_multiple_truth])
    ax1.set_title("Ground Truth")
    ax1.set_ylabel("Count")

    # Predictions
    ax2.bar(["Individual", "Multiple"], [cat_type_individual_full, cat_type_multiple_full])
    ax2.set_title("Predictions")
    ax2.set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "deity_type_comparison.png"))
    plt.show()

## Certainty vs Match Score Correlation

In [ ]:
# Analyze correlation between certainty and match score
certainty_score_pairs = []

for _, row in results_df.iterrows():
    certs = row.get("certainty_deity", "")
    scores = row.get("score", [])

    if not certs or not scores:
        continue

    try:
        certainty_list = [float(x.strip()) for x in str(certs).split(",") if x.strip()]
        if isinstance(scores, str):
            scores = ast.literal_eval(scores)
        if len(certainty_list) != len(scores):
            continue
        certainty_score_pairs.extend(zip(certainty_list, scores))
    except Exception:
        continue

if certainty_score_pairs:
    certainties, scores = zip(*certainty_score_pairs)
    correlation = np.corrcoef(certainties, scores)[0, 1]

    print("\nCertainty vs Match Score:")
    print("=" * 80)
    print(f"Correlation: {correlation:.3f}")

    # Count frequencies
    pair_counts = Counter(zip(certainties, scores))

    # Prepare plot data
    x_vals, y_vals, sizes = [], [], []
    for (x, y), count in pair_counts.items():
        x_vals.append(x)
        y_vals.append(y)
        sizes.append(count * 40)

    # Plot
    plt.figure(figsize=(10, 8))
    plt.scatter(x_vals, y_vals, s=sizes, alpha=0.6, edgecolor="black")

    # Annotate counts
    for x, y, count in zip(x_vals, y_vals, pair_counts.values()):
        if count > 1:
            plt.text(x, y, str(count), fontsize=9, ha="center", va="center")

    plt.title(f"Certainty vs. Match Score (Point Size = Frequency)\nCorrelation: {correlation:.3f}")
    plt.xlabel("Certainty (Deity)")
    plt.ylabel("Match Score")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "certainty_vs_score.png"))
    plt.show()
else:
    print("No valid certainty-score pairs found")

## Regional Analysis (if region data available)

In [ ]:
# Check if region columns exist
if "Region" in truth_df.columns and "Subregion" in truth_df.columns:
    # Merge region info into results
    meta_cols = ["uuid", "Region", "Subregion"]
    meta_df = truth_df[meta_cols].drop_duplicates()
    results_with_region = results_df.merge(meta_df, on="uuid", how="left")

    # Calculate TP, FP, FN per row
    results_with_region[["TP", "FP", "FN"]] = results_with_region.apply(
        calculate_fp_fn_tp, axis=1, result_type="expand"
    )

    # Generate regional plots
    plot_regional_metrics(
        match_df=results_with_region,
        output_dir=OUTPUT_DIR,
        region_col="Region",
        subregion_col="Subregion",
    )

    print("\nRegional analysis complete. Plots saved to output directory.")

    # Display regional summaries
    print("\nFalse Positives by Region:")
    fp_region = results_with_region.groupby("Region")["FP"].sum().sort_values(ascending=False)
    print(fp_region)

    print("\nFalse Negatives by Region:")
    fn_region = results_with_region.groupby("Region")["FN"].sum().sort_values(ascending=False)
    print(fn_region)
else:
    print("Region columns not found in ground truth data")

## Summary Statistics

In [ ]:
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print("\n1. DEITY IDENTIFICATION:")
print(f"   - Ground truth deities: {num_deities_total}")
print(f"   - Predicted deities: {total_deities_in_full_df}")
print(f"   - Successfully matched: {num_deities_matched}")
print(f"   - Match rate: {num_deities_matched / num_deities_total * 100:.2f}%")

print("\n2. PERFORMANCE METRICS:")
print(f"   - Precision: {overall_metrics['Precision']}")
print(f"   - Recall: {overall_metrics['Recall']}")
print(f"   - F1 Score: {overall_metrics['F1']}")
print(f"   - Accuracy: {overall_metrics['Accuracy']}")

if certainty_score_pairs:
    print("\n3. CERTAINTY CORRELATION:")
    print(f"   - Correlation coefficient: {correlation:.3f}")

print("\n4. OUTPUT FILES:")
print(f"   All results and visualizations saved to: {OUTPUT_DIR}")
print("\n" + "=" * 80)